# GO Reasoning Dataset

### Step 1: Make a fasta file of all proteins in our train + test set

In [1]:
import pandas as pd
import numpy as np
import os
from datasets import load_dataset

In [2]:
ds = load_dataset("wanglab/cafa5", "cafa5_reasoning")
cafa5_train = ds["train"].to_pandas()
cafa5_test = ds["test"].to_pandas()
cafa5_test_super = ds["test_superset"].to_pandas()

all = pd.concat([cafa5_train, cafa5_test, cafa5_test_super])
all = all.drop(columns=['protein_names','protein_function','organism','length','subcellular_location','go_ids','go_bp','go_mf','go_cc','string_id','interaction_partners','full_interaction_info','interpro_ids','interpro_location','structure_path'])
all.drop_duplicates(inplace=True)
all.reset_index(drop=True, inplace=True)

README.md: 0.00B [00:00, ?B/s]

cafa5_reasoning/train-00000-of-00002.par(…):   0%|          | 0.00/104M [00:00<?, ?B/s]

cafa5_reasoning/train-00001-of-00002.par(…):   0%|          | 0.00/77.7M [00:00<?, ?B/s]

cafa5_reasoning/test-00000-of-00001.parq(…):   0%|          | 0.00/393k [00:00<?, ?B/s]

cafa5_reasoning/test_superset-00000-of-0(…):   0%|          | 0.00/155M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/133534 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/264 [00:00<?, ? examples/s]

Generating test_superset split:   0%|          | 0/141847 [00:00<?, ? examples/s]

In [3]:
all

,protein_id,sequence
0,A0A087X1C5,MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNL...
1,A0A0B4J2F0,MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...
2,A0A0A7EPL0,MVIPATSRFGFRAEFNTKEFQASCISLANEIDAAIGRNEVPGNIQE...
3,A0A0B4J1G0,MWQLLLPTALVLTAFSGIQAGLQKAVVNLDPKWVRVLEEDSVTLRC...
4,A0A0B4J1N3,MRLLALSGLLCMLLLCFCIFSSEGRRHPAKSLKLRRCCHLSPRSKL...
...,...,...
201752,Q4A3R3,MGTSAVILEICLLLSQVLTTVSSTTQTESTTEDRTQITETAFWETQ...
201753,Q5E9L2,MNWRFVELLYFLFVWGRISVQPSHQEPAATDQHVSKEFDWLISDRG...
201754,Q3T0K9,MNSKGQYPTQPTYPVQPPGNPVYPQTLHLPQAPPYTDAPPAYSELY...
201755,Q58CX7,MEPRLGPKAAALHLGWPFLLLWVSGLSYSVSSPASPSPSPVSRVRT...


In [4]:
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO

# Convert DataFrame rows to SeqRecord objects
records = [
    SeqRecord(Seq(row['sequence']), id=row['protein_id'], description="")
    for _, row in all.iterrows()
]

# Write to FASTA file
with open("all_proteins.fasta", "w") as output_handle:
    SeqIO.write(records, output_handle, "fasta")

### Step 2: BLAST these proteins against nr and get the top 10 hits

In [ ]:
time diamond blastp \
  -q all_proteins.fasta \
  -d <path_to_database>/nr.dmnd  \
  --masking 0 \
  --sensitive -k 25 \
  -b 10 -c1 \
  --unal 1 \
  --threads 64 \
  -f 6 qseqid  qstart qend qlen qstrand \
       sseqid  sstart send slen \
       pident evalue \
       staxids sscinames \
       stitle \
  -o all_cafa_proteins.tsv

In [1]:
import pandas as pd
import numpy as np

In [4]:
all_cafa_proteins = pd.read_csv('all_cafa_proteins.tsv', sep='\t', 
                                names=['cafa5_protein', 'q_start', 'q_end','q_length', 'strand',
                                      'blast_protein','t_start','t_end', 't_length','pident','evalue','taxid','species','protein_name'])

In [5]:
all_cafa_proteins

,cafa5_protein,q_start,q_end,q_length,strand,blast_protein,t_start,t_end,t_length,pident,evalue,taxid,species,protein_name
0,A0A087X1C5,1,515,515,+,A0A087X1C5.1,1,515,515,100.0,0.0,9606.0,Homo sapiens,A0A087X1C5.1 PUTATIVE PSEUDOGENE: RecName: Ful...
1,A0A087X1C5,1,515,515,+,NP_001335315.1,1,516,516,99.8,0.0,9606.0,Homo sapiens,NP_001335315.1 putative cytochrome P450 2D7 [H...
2,A0A087X1C5,1,515,515,+,AAO49806.1,1,516,516,98.6,0.0,9606.0,Homo sapiens,AAO49806.1 cytochrome P450 [Homo sapiens]
3,A0A087X1C5,1,515,515,+,AEU08335.1,1,497,497,92.0,0.0,9606.0,Homo sapiens,AEU08335.1 CYP2D6 [Homo sapiens]
4,A0A087X1C5,1,515,515,+,XP_034805195.3,1,497,497,92.2,0.0,9597.0,Pan paniscus,XP_034805195.3 cytochrome P450 2D6 isoform X1 ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4940545,A7E300,34,1112,1112,+,XP_027386041.1,445,1523,1523,99.5,0.0,30522.0,Bos indicus x Bos taurus,XP_027386041.1 rho GTPase-activating protein 7...
4940546,A7E300,33,1112,1112,+,XP_027386044.1,84,1163,1163,99.4,0.0,30522.0,Bos indicus x Bos taurus,XP_027386044.1 rho GTPase-activating protein 7...
4940547,A7E300,37,1112,1112,+,ELR59854.1,1,1076,1076,99.8,0.0,72004.0,Bos mutus,"ELR59854.1 Rho GTPase-activating protein 7, pa..."
4940548,A7E300,34,1112,1112,+,XP_010837132.1,445,1523,1523,99.4,0.0,43346.0,Bison bison bison,XP_010837132.1 PREDICTED: rho GTPase-activatin...


In [6]:
all_cafa_proteins['protein_name'] = all_cafa_proteins['protein_name'].str.split('>').str[0].str.strip()

In [7]:
from datasets import Dataset

# Create Hugging Face Datasets
blast_results = Dataset.from_pandas(all_cafa_proteins, preserve_index=False)

tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
from datasets import DatasetDict

blast_dataset = DatasetDict({
    "metadata": blast_results
})

In [9]:
blast_dataset

DatasetDict({
    metadata: Dataset({
        features: ['cafa5_protein', 'q_start', 'q_end', 'q_length', 'strand', 'blast_protein', 't_start', 't_end', 't_length', 'pident', 'evalue', 'taxid', 'species', 'protein_name'],
        num_rows: 4940550
    })
})

In [ ]:
blast_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="blast_metadata",
    commit_message="Uploading the BLAST results. 25 proteins per cafa5_protein"
)

### Step 3: Format the PPI Data

Currently the PPI data does not include the Protein names, so I need to include that information when we are creating the data for the model

Download the dataset with all of the info about the proteins

In [ ]:
wget https://stringdb-downloads.org/download/protein.info.v12.0.txt.gz

In [ ]:
gunzip protein.info.v12.0.txt.gz

In [ ]:
wget https://string-db.org/mapping_files/uniprot/all_organisms.uniprot_2_string.tsv

Get the PPI Partners

In [2]:
import pandas as pd
import numpy as np
import os
from datasets import load_dataset

In [3]:
ds = load_dataset("wanglab/cafa5", "cafa5_reasoning")
cafa5_train = ds["train"].to_pandas()
cafa5_test = ds["test"].to_pandas()
cafa5_test_super = ds["test_superset"].to_pandas()

In [93]:
cafa5_train['species_string'] = cafa5_train['string_id'].apply(
    lambda x: x.split('.')[0] if isinstance(x, str) and '.' in x else None
)
cafa5_test['species_string'] = cafa5_test['string_id'].apply(
    lambda x: x.split('.')[0] if isinstance(x, str) and '.' in x else None
)
cafa5_test_super['species_string'] = cafa5_test_super['string_id'].apply(
    lambda x: x.split('.')[0] if isinstance(x, str) and '.' in x else None
)

Load in STRING Data

In [5]:
string_info = pd.read_csv('<path_to_string_data>/protein.info.v12.0.txt', sep='\t')

In [6]:
string_info['species'] = string_info['#string_protein_id'].str.split('.').str[0]
string_info['string_name'] = string_info['annotation'].apply(lambda x: x.split('; ', 1)[0])
string_info['string_description'] = string_info['annotation'].apply(lambda x: x.split('; ', 1)[1] if '; ' in x else None)

In [7]:
string_info[100:110]

,#string_protein_id,preferred_name,protein_size,annotation,species,string_name,string_description
100,23.BEL05_00560,OEG74124.1,168,Hypothetical protein; Derived by automated com...,23,Hypothetical protein,Derived by automated computational analysis us...
101,23.BEL05_00565,OEG74125.1,1003,Collagenase; Derived by automated computationa...,23,Collagenase,Derived by automated computational analysis us...
102,23.BEL05_00570,OEG74126.1,338,NAD(P)H-quinone oxidoreductase; Derived by aut...,23,NAD(P)H-quinone oxidoreductase,Derived by automated computational analysis us...
103,23.BEL05_00575,OEG74127.1,142,Hypothetical protein; Derived by automated com...,23,Hypothetical protein,Derived by automated computational analysis us...
104,23.BEL05_00580,OEG74128.1,345,Galactose mutarotase; Converts alpha-aldose to...,23,Galactose mutarotase,Converts alpha-aldose to the beta-anomer.
105,23.BEL05_00585,galK,388,Galactokinase; Catalyzes the transfer of the g...,23,Galactokinase,Catalyzes the transfer of the gamma-phosphate ...
106,23.BEL05_00590,OEG74130.1,347,Galactose-1-phosphate uridylyltransferase; Der...,23,Galactose-1-phosphate uridylyltransferase,Derived by automated computational analysis us...
107,23.BEL05_00595,OEG74131.1,475,Sodium:solute symporter; Derived by automated ...,23,Sodium:solute symporter,Derived by automated computational analysis us...
108,23.BEL05_00600,OEG74132.1,1055,Beta-galactosidase; Derived by automated compu...,23,Beta-galactosidase,Derived by automated computational analysis us...
109,23.BEL05_00605,OEG74282.1,310,Hypothetical protein; Derived by automated com...,23,Hypothetical protein,Derived by automated computational analysis us...


Uniprot to STRING mapping

In [8]:
import pandas as pd

mapping_path = "<path_to_string_data>/all_organisms.uniprot_2_string.tsv"
cols = ["species", "uniprot_ac|uniprot_id", "string_id", "identity", "bit_score"]

# Load the file, skipping comment lines (those starting with '#') and assigning custom column names
uniprot_string_map_df = pd.read_csv(
    mapping_path,
    sep="\t",
    comment="#",
    header=None,
    names=cols,
    on_bad_lines='skip',
    engine='python'
)

# Split the combined UniProt column into two: accession and ID
uniprot_string_map_df[['uniprot_ac', 'uniprot_id']] = uniprot_string_map_df['uniprot_ac|uniprot_id'].str.split('|', n=1, expand=True)

# Drop the original combined column
uniprot_string_map_df.drop(columns=['uniprot_ac|uniprot_id'], inplace=True)

# Preview (optional)
# print(uniprot_string_map_df.head())

Create the extra metadata row in 'all'

In [9]:
uniprot_string_map_df

,species,string_id,identity,bit_score,uniprot_ac,uniprot_id
0,5874,5874.P16019,Src,Src,P16019,HSP70_THEAN
1,5874,5874.P25781,Src,Src,P25781,CYSP_THEAN
2,5874,5874.Q26671,Src,Src,Q26671,CDC2H_THEAN
3,5874,5874.Q27399,Src,Src,Q27399,Q27399_THEAN
4,5874,5874.Q37679,Src,Src,Q37679,COX3_THEAN
...,...,...,...,...,...,...
46131357,1030841,1030841.HMPREF9370_0777,100.0,198.0,G4CNW8,G4CNW8_9NEIS
46131358,1030841,1030841.HMPREF9370_0925,100.0,419.5,G4CPB5,G4CPB5_9NEIS
46131359,1030841,1030841.HMPREF9370_1435,100.0,771.9,G4CQS5,G4CQS5_9NEIS
46131360,1030841,1030841.HMPREF9370_1692,100.0,592.8,G4CRI2,G4CRI2_9NEIS


In [10]:
import numpy as np
import pandas as pd
from tqdm import tqdm

# Fast build: (species, preferred_name) -> string_id
name_to_string_id_lookup = {
    (str(species), preferred_name): string_id
    for species, preferred_name, string_id in zip(
        string_info['species'],
        string_info['preferred_name'],
        string_info['#string_protein_id']
    )
}

# Fast build: (species, string_id) -> UniProt metadata
string_id_to_uniprot_metadata = {
    (str(species), string_id): {
        'string_protein_id': string_id,
        'uniprot_ac': uniprot_ac,
        'uniprot_id': uniprot_id,
        'identity': identity,
        'bit_score': bit_score
    }
    for species, string_id, uniprot_ac, uniprot_id, identity, bit_score in zip(
        uniprot_string_map_df['species'],
        uniprot_string_map_df['string_id'],
        uniprot_string_map_df['uniprot_ac'],
        uniprot_string_map_df['uniprot_id'],
        uniprot_string_map_df['identity'],
        uniprot_string_map_df['bit_score']
    )
}

In [90]:
type(cafa5_train['full_interaction_info'][2])

numpy.ndarray

In [94]:
def collect_metadata_flat(df):
    metadata_rows = []

    valid_mask = df['interaction_partners'].apply(lambda x: isinstance(x, np.ndarray))
    valid_indices = df.index[valid_mask]

    for idx in tqdm(valid_indices, desc="Collecting UniProt metadata"):
        partners = df.at[idx, 'interaction_partners']
        interaction_info = df.at[idx, 'full_interaction_info']
        species = str(df.at[idx, 'species_string'])
        cafa_protein = df.at[idx, 'protein_id']

        # Safety check: make sure the interaction_info is a list and matches the partner list
        if not isinstance(interaction_info, np.ndarray) or len(interaction_info) != len(partners):
            print("something gone wrong at ",idx)

        for partner, info in zip(partners, interaction_info):
            string_id = name_to_string_id_lookup.get((species, partner))
            if string_id:
                uniprot_metadata = string_id_to_uniprot_metadata.get((species, string_id))
                if uniprot_metadata:
                    metadata_rows.append({
                        'cafa5_protein': cafa_protein,
                        'partner_id': partner,
                        **uniprot_metadata,
                        **info  # includes combined_score, coexpression, experimental, etc.
                    })

    return pd.DataFrame(metadata_rows)

In [95]:
train_metadata_df = collect_metadata_flat(cafa5_train)
test_metadata_df = collect_metadata_flat(cafa5_test)
super_metadata_df = collect_metadata_flat(cafa5_test_super)

In [96]:
# Optionally combine all:
all_metadata_df = pd.concat([train_metadata_df, test_metadata_df, super_metadata_df], ignore_index=True)
all_metadata_df = all_metadata_df.drop_duplicates()

In [97]:
all_metadata_df

,cafa5_protein,partner_id,string_protein_id,uniprot_ac,uniprot_id,identity,bit_score,coexpression,combined_score,cooccurence,database,experimental,fusion,neighborhood,protein2,textmining
0,A0A0A7EPL0,ESD4,3702.Q94F30,Q94F30,ESD4_ARATH,Src,Src,0,823,0,0,107,0,0,ESD4,811
1,A0A0A7EPL0,SUMO1,3702.P55852,P55852,SUMO1_ARATH,Src,Src,0,958,0,415,558,0,0,SUMO1,854
2,A0A0A7EPL0,ULP1D,3702.Q2PS26,Q2PS26,ULP1D_ARATH,Src,Src,0,809,0,0,107,0,0,ULP1D,795
3,A0A0A7EPL0,ULP1B,3702.O65278,O65278,ULP1B_ARATH,Src,Src,0,809,0,0,107,0,0,ULP1B,795
4,A0A0A7EPL0,SUMO8,3702.B3H5R8,B3H5R8,SUMO8_ARATH,Src,Src,0,850,0,415,558,0,0,SUMO8,469
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6156760,Q5E9L2,GPR26,9913.ENSBTAP00000071556,A0A3Q1MBU2,A0A3Q1MBU2_BOVIN,100.0,658.3,96,713,0,0,0,0,0,GPR26,697
6156761,Q5E9L2,ASTN1,9913.ENSBTAP00000053268,F1MLR3,F1MLR3_BOVIN,100.0,2680.6,100,707,0,0,0,0,0,ASTN1,688
6156762,Q5E9L2,SIRT1,9913.ENSBTAP00000018630,F1MQB8,F1MQB8_BOVIN,100.0,1379.8,0,729,0,0,585,0,0,SIRT1,376
6156763,A7E300,TNS2,9913.ENSBTAP00000067960,A0A3Q1MBB1,A0A3Q1MBB1_BOVIN,100.0,2906.3,0,895,0,0,818,0,0,TNS2,450


### Testing what are the PPI partners that I am losing

In [15]:
all = pd.concat([cafa5_train, cafa5_test, cafa5_test_super])
all.reset_index(drop=True, inplace=True)

In [16]:
import numpy as np

# Only keep rows where interaction_partners is a list
valid_mask = all['interaction_partners'].apply(lambda x: isinstance(x, np.ndarray))
filtered_df = all.loc[valid_mask, ['protein_id', 'interaction_partners']]

# Flatten using list comprehension
partner_pairs = [
    (protein_id, partner)
    for protein_id, partners in zip(filtered_df['protein_id'], filtered_df['interaction_partners'])
    for partner in partners
]

# Convert to DataFrame
partner_pairs_df = pd.DataFrame(partner_pairs, columns=['cafa5_protein', 'partner_id'])
partner_pairs_df = partner_pairs_df.drop_duplicates()

In [17]:
# Step 2: Check which are missing from all_metadata_df
# Use an inner merge to find matches
merged = partner_pairs_df.merge(
    all_metadata_df[['cafa5_protein', 'partner_id']],
    on=['cafa5_protein', 'partner_id'],
    how='left',
    indicator=True
)

# Filter missing entries
missing = merged[merged['_merge'] == 'left_only']

# Summary
print(f"Total partner pairs in 'all': {len(partner_pairs_df)}")
print(f"Found in metadata: {len(partner_pairs_df) - len(missing)}")
print(f"Missing from metadata: {len(missing)}")

# Optional: view missing entries
# print(missing.head())

Total partner pairs in 'all': 4044140
Found in metadata: 4000111
Missing from metadata: 44029


In [18]:
missing

,cafa5_protein,partner_id,_merge
266,A0A089QRB9,pks3,left_only
1468,A0A096MKF8,Med22,left_only
1859,A0A0B4K6G3,His3.3B,left_only
1860,A0A0B4K6G3,His3:CG33806,left_only
1863,A0A0B4K6G3,His3:CG33809,left_only
...,...,...,...
4043759,Q9NQZ3,CDY2B,left_only
4043760,Q9NQZ3,BPY2C,left_only
4043761,Q9NQZ3,VCY,left_only
4043775,Q9NQZ3,DAZ2,left_only


In [19]:
# Step 1: Count total partners per cafa5_protein in `all`
partner_counts_all = (
    all[all['interaction_partners'].apply(lambda x: isinstance(x, np.ndarray))]
    .assign(total_partners=lambda df: df['interaction_partners'].apply(len))
    [['protein_id', 'total_partners']]
    .rename(columns={'protein_id': 'cafa5_protein'})
)
partner_counts_all.drop_duplicates()
len(partner_counts_all)

# Step 2: Count number of missing partners per protein
missing_counts = (
    missing.groupby('cafa5_protein')
    .size()
    .reset_index(name='missing_partners')
)

# Step 3: Merge both counts
summary_df = partner_counts_all.merge(missing_counts, on='cafa5_protein', how='inner')

# Optional: Sort by number of missing partners
summary_df = summary_df.sort_values(by='total_partners', ascending=False)

summary_df.head()

,cafa5_protein,total_partners,missing_partners
5715,B6V6V6,1324,1126
10958,A9L913,792,28
2369,P62986,773,58
8132,P62986,773,58
12560,P04637,766,25


In [20]:
summary_df

,cafa5_protein,total_partners,missing_partners
5715,B6V6V6,1324,1126
10958,A9L913,792,28
2369,P62986,773,58
8132,P62986,773,58
12560,P04637,766,25
...,...,...,...
6660,P0C027,1,1
12415,Q9Y561,1,1
12507,A0A087WUL8,1,1
4940,P59991,1,1


In [21]:
summary_df['diff'] = summary_df['total_partners'] - summary_df['missing_partners']

In [22]:
len(summary_df[summary_df['diff'] == 0])

55

In [23]:
summary_df.sort_values(by='diff', ascending=False)

,cafa5_protein,total_partners,missing_partners,diff
10958,A9L913,792,28,764
12560,P04637,766,25,741
755,P04637,766,25,741
2369,P62986,773,58,715
8132,P62986,773,58,715
...,...,...,...,...
5775,P0C027,1,1,0
11335,Q5R412,1,1,0
5728,E9PI22,1,1,0
12561,E9PI22,1,1,0


55 protein out of 275k will be losing all of their PPI partners. I am fine with that

## Fetching Uniprot info

In [60]:
import requests
import pandas as pd
from time import sleep
from io import StringIO
from tqdm import tqdm

def fetch_uniprot_metadata_large(all_accessions, output_file="ppi_uniprot_metadata.tsv", deleted_file="ppi_deleted_accessions.txt", batch_size=500):
    base_url = "https://rest.uniprot.org/uniprotkb/search"
    fields = [
        "accession",
        "protein_name",
        "cc_function",
        "cc_subcellular_location"
    ]

    # Prepare output
    first_batch = True
    deleted_accessions = []

    # Break into batches
    batches = [all_accessions[i:i + batch_size] for i in range(0, len(all_accessions), batch_size)]

    for batch in tqdm(batches, desc="Processing batches"):
        query = " OR ".join([f"accession:{acc}" for acc in batch])
        params = {
            "query": query,
            "format": "tsv",
            "fields": ",".join(fields),
            "size": batch_size
        }

        try:
            r = requests.get(base_url, params=params, timeout=30)
            if r.status_code != 200:
                print(f"Failed with status {r.status_code} — retrying after delay")
                sleep(5)
                continue

            df = pd.read_csv(StringIO(r.text), sep="\t")

            # Identify deleted entries
            deleted_mask = df["Protein names"].str.contains("deleted", case=False, na=False)
            deleted_accessions.extend(df.loc[deleted_mask, "Entry"].tolist())

            # Filter non-deleted
            df_clean = df[~deleted_mask]

            # Write to file incrementally
            df_clean.to_csv(output_file, sep="\t", index=False, mode='a', header=first_batch)
            first_batch = False

        except Exception as e:
            print(f"Error on batch: {e}")
            sleep(5)

        sleep(0.1)  # rate-limit buffer

    # Save deleted accessions
    with open(deleted_file, "w") as f:
        for acc in deleted_accessions:
            f.write(acc + "\n")

    print(f"✅ Done. Saved to {output_file}, deleted proteins saved to {deleted_file}")

In [61]:
fetch_uniprot_metadata_large(all_metadata_df['uniprot_ac'].unique(), 
                             output_file="ppi_uniprot_metadata.tsv", deleted_file="ppi_deleted_accessions.txt")

Processing batches: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 656/656 [25:42<00:00,  2.35s/it]

✅ Done. Saved to ppi_uniprot_metadata.tsv, deleted proteins saved to ppi_deleted_accessions.txt


In [24]:
uniprot = pd.read_csv('ppi_uniprot_metadata.tsv', sep='\t')
uniprot = uniprot.rename(columns={'Entry' : 'uniprot_ac'})

In [25]:
uniprot

,uniprot_ac,Protein names,Function [CC],Subcellular location [CC]
0,A1Z8P9,Nucleoprotein TPR (Nuclear envelope antigen Bx...,FUNCTION: Component of the nuclear pore comple...,SUBCELLULAR LOCATION: Nucleus {ECO:0000269|Pub...
1,G5EDN3,Protein timeless homolog,FUNCTION: Plays an important role in chromosom...,SUBCELLULAR LOCATION: Nucleus {ECO:0000269|Pub...
2,P08508,Low affinity immunoglobulin gamma Fc region re...,FUNCTION: Receptor for the Fc region of comple...,SUBCELLULAR LOCATION: Cell membrane {ECO:00002...
3,P20491,High affinity immunoglobulin epsilon receptor ...,FUNCTION: Adapter protein containing an immuno...,SUBCELLULAR LOCATION: Cell membrane {ECO:00002...
4,P27870,Proto-oncogene vav (p95vav),FUNCTION: Couples tyrosine kinase signals with...,NaN
...,...,...,...,...
301565,Q8ZQV3,Glycosyl transferase,NaN,NaN
301566,Q8ZQZ0,Cytoplasmic protein,NaN,NaN
301567,Q8ZRA8,Acetyltransferase (EC 2.3.1.-),NaN,NaN
301568,Q8ZS13,LysR family transcriptional regulator,NaN,NaN


## Merging it with the PPI data

In [98]:
all_metadata_df = all_metadata_df.merge(uniprot, on='uniprot_ac', how='left')

In [99]:
all_metadata_df[all_metadata_df['Protein names'] == 'merged']

,cafa5_protein,partner_id,string_protein_id,uniprot_ac,uniprot_id,identity,bit_score,coexpression,combined_score,cooccurence,database,experimental,fusion,neighborhood,protein2,textmining,Protein names,Function [CC],Subcellular location [CC]
458,A0A0B4K6M2,nSyb,7227.FBpp0303878,M9PE71,M9PE71_DROME,100.0,300.1,193,796,0,500,0,0,0,nSyb,537,merged,NaN,NaN
500,A0A0B4K774,dock,7227.FBpp0077679,Q9VPU1,Q9VPU1_DROME,100.0,977.6,95,829,0,500,245,0,0,dock,561,merged,NaN,NaN
534,A0A0B4K7K9,nSyb,7227.FBpp0303878,M9PE71,M9PE71_DROME,100.0,300.1,100,868,0,0,47,0,0,nSyb,859,merged,NaN,NaN
540,A0A0B4K823,dock,7227.FBpp0077679,Q9VPU1,Q9VPU1_DROME,100.0,977.6,0,781,0,0,708,0,0,dock,283,merged,NaN,NaN
689,A0A0B4KEU2,sba,7227.FBpp0303406,A0A0B4KHZ2,A0A0B4KHZ2_DROME,100.0,2352.4,52,713,0,0,0,0,0,sba,710,merged,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3996476,Q8I2J3,PF3D7_1311800,36329.Q8IEK1,Q8IEK1,Q8IEK1_PLAF7,Src,Src,42,719,0,0,0,0,0,PF3D7_1311800,719,merged,NaN,NaN
3999434,B4F7F3,Hnrnpab,10116.ENSRNOP00000004885,Q9QX81,Q9QX81_RAT,100.0,565.8,0,722,0,720,0,0,0,Hnrnpab,50,merged,NaN,NaN
3999453,Q63744,Rhoc,10116.ENSRNOP00000073774,A0A0G2K6E0,A0A0G2K6E0_RAT,100.0,433.3,69,769,0,400,275,0,0,Rhoc,498,merged,NaN,NaN
3999521,Q76CY5,dnd1,8364.ENSXETP00000038374,Q28I69,Q28I69_XENTR,100.0,743.4,106,747,0,0,117,0,0,dnd1,706,merged,NaN,NaN


In [100]:
string_info[string_info['#string_protein_id']=='7227.FBpp0303878']

,#string_protein_id,preferred_name,protein_size,annotation,species,string_name,string_description
4362277,7227.FBpp0303878,nSyb,206,"Neuronal synaptobrevin, isoform J; Neuronal Sy...",7227,"Neuronal synaptobrevin, isoform J",Neuronal Synaptobrevin (nSyb) encodes a SNAP r...


In [ ]:
from tqdm import tqdm
import pandas as pd

# Step 1: Build lookup dictionaries from string_info
name_lookup = dict(zip(string_info['#string_protein_id'], string_info['string_name']))
desc_lookup = dict(zip(string_info['#string_protein_id'], string_info['string_description']))

In [101]:
# Step 2: Mask rows with missing or 'merged' protein names
mask = (all_metadata_df['Protein names'].isna()) | (all_metadata_df['Protein names'] == 'merged')
indices = all_metadata_df[mask].index

# Step 3: Update those rows in-place using tqdm for a progress bar
for idx in tqdm(indices, desc="Updating protein names and descriptions"):
    protein_id = all_metadata_df.at[idx, 'string_protein_id']
    
    if protein_id in name_lookup:
        all_metadata_df.at[idx, 'Protein names'] = name_lookup[protein_id]
    if protein_id in desc_lookup:
        all_metadata_df.at[idx, 'Function [CC]'] = desc_lookup[protein_id]

Updating protein names and descriptions: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 172851/172851 [00:06<00:00, 25429.49it/s]


In [102]:
indices[100:110]

Index([10306, 10308, 10310, 10314, 10358, 10359, 10369, 10386, 10390, 10397], dtype='int64')

In [103]:
all_metadata_df.loc[3996476]

cafa5_protein                                          Q8I2J3
partner_id                                      PF3D7_1311800
string_protein_id                                36329.Q8IEK1
uniprot_ac                                             Q8IEK1
uniprot_id                                       Q8IEK1_PLAF7
identity                                                  Src
bit_score                                                 Src
coexpression                                               42
combined_score                                            719
cooccurence                                                 0
database                                                    0
experimental                                                0
fusion                                                      0
neighborhood                                                0
protein2                                        PF3D7_1311800
textmining                                                719
Protein 

In [104]:
all_metadata_df[all_metadata_df['Protein names'].isna()]

,cafa5_protein,partner_id,string_protein_id,uniprot_ac,uniprot_id,identity,bit_score,coexpression,combined_score,cooccurence,database,experimental,fusion,neighborhood,protein2,textmining,Protein names,Function [CC],Subcellular location [CC]


In [105]:
all_metadata_df

,cafa5_protein,partner_id,string_protein_id,uniprot_ac,uniprot_id,identity,bit_score,coexpression,combined_score,cooccurence,database,experimental,fusion,neighborhood,protein2,textmining,Protein names,Function [CC],Subcellular location [CC]
0,A0A0A7EPL0,ESD4,3702.Q94F30,Q94F30,ESD4_ARATH,Src,Src,0,823,0,0,107,0,0,ESD4,811,Ubiquitin-like-specific protease ESD4 (EC 3.4....,FUNCTION: Protease that catalyzes two essentia...,SUBCELLULAR LOCATION: Nucleus membrane {ECO:00...
1,A0A0A7EPL0,SUMO1,3702.P55852,P55852,SUMO1_ARATH,Src,Src,0,958,0,415,558,0,0,SUMO1,854,Small ubiquitin-related modifier 1 (AtSUMO1) (...,FUNCTION: Ubiquitin-like protein which can be ...,SUBCELLULAR LOCATION: Nucleus {ECO:0000269|Pub...
2,A0A0A7EPL0,ULP1D,3702.Q2PS26,Q2PS26,ULP1D_ARATH,Src,Src,0,809,0,0,107,0,0,ULP1D,795,Ubiquitin-like-specific protease 1D (EC 3.4.22...,FUNCTION: Protease that catalyzes two essentia...,SUBCELLULAR LOCATION: Nucleus speckle {ECO:000...
3,A0A0A7EPL0,ULP1B,3702.O65278,O65278,ULP1B_ARATH,Src,Src,0,809,0,0,107,0,0,ULP1B,795,Putative ubiquitin-like-specific protease 1B (...,FUNCTION: Protease that catalyzes two essentia...,NaN
4,A0A0A7EPL0,SUMO8,3702.B3H5R8,B3H5R8,SUMO8_ARATH,Src,Src,0,850,0,415,558,0,0,SUMO8,469,Putative small ubiquitin-related modifier 8 (A...,FUNCTION: Ubiquitin-like protein which can be ...,SUBCELLULAR LOCATION: Nucleus {ECO:0000250}. C...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4000106,Q5E9L2,GPR26,9913.ENSBTAP00000071556,A0A3Q1MBU2,A0A3Q1MBU2_BOVIN,100.0,658.3,96,713,0,0,0,0,0,GPR26,697,G-protein coupled receptor 26,FUNCTION: Orphan receptor. Displays a signific...,SUBCELLULAR LOCATION: Cell membrane {ECO:00002...
4000107,Q5E9L2,ASTN1,9913.ENSBTAP00000053268,F1MLR3,F1MLR3_BOVIN,100.0,2680.6,100,707,0,0,0,0,0,ASTN1,688,Astrotactin 1,NaN,NaN
4000108,Q5E9L2,SIRT1,9913.ENSBTAP00000018630,F1MQB8,F1MQB8_BOVIN,100.0,1379.8,0,729,0,0,585,0,0,SIRT1,376,protein acetyllysine N-acetyltransferase (EC 2...,NaN,SUBCELLULAR LOCATION: Nucleus {ECO:0000256|ARB...
4000109,A7E300,TNS2,9913.ENSBTAP00000067960,A0A3Q1MBB1,A0A3Q1MBB1_BOVIN,100.0,2906.3,0,895,0,0,818,0,0,TNS2,450,Tensin-2 (EC 3.1.3.48) (C1 domain-containing p...,NaN,"SUBCELLULAR LOCATION: Cell junction, focal adh..."


## Edit the cafa5 data to reflect this update

In [ ]:
cafa5_train = cafa5_train.drop(columns=['species_string','interaction_partners_metadata','full_interaction_info'])
cafa5_test = cafa5_test.drop(columns=['species_string','interaction_partners_metadata','full_interaction_info'])
cafa5_test_super = cafa5_test_super.drop(columns=['species_string','interaction_partners_metadata','full_interaction_info'])

In [110]:
# Step 1: Group partner_ids by cafa5_protein
grouped_partners = (
    all_metadata_df
    .groupby('cafa5_protein')['partner_id']
    .apply(list)
)

In [111]:
# Step 2: Assign
cafa5_train['interaction_partners'] = cafa5_train['protein_id'].map(grouped_partners)
cafa5_test['interaction_partners'] = cafa5_test['protein_id'].map(grouped_partners)
cafa5_test_super['interaction_partners'] = cafa5_test_super['protein_id'].map(grouped_partners)

In [112]:
cafa5_train

,protein_id,protein_names,protein_function,organism,length,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc,structure_path,string_id,interaction_partners,interpro_ids,interpro_location
0,A0A087X1C5,Putative cytochrome P450 2D7 (EC 1.14.14.1),May be responsible for the metabolism of many ...,Homo sapiens (Human),515.0,Membrane {ECO:0000305}; Multi-pass membrane pr...,MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNL...,"[GO:0008152, GO:0051716, GO:0009987, GO:007146...","[GO:0008150, GO:0008152, GO:0009987, GO:005089...","[GO:0003674, GO:0003824, GO:0016491, GO:001670...","[GO:0005575, GO:0110165, GO:0005622, GO:004322...",AF-A0A087X1C5-F1-model_v4.cif,None,NaN,"[IPR036396, IPR017972, IPR001128, IPR002401, I...","{""IPR036396"": [24, 515], ""IPR050182"": [13, 494..."
1,A0A0B4J2F0,Protein PIGBOS1 (PIGB opposite strand protein 1),Plays a role in regulation of the unfolded pro...,Homo sapiens (Human),54.0,Mitochondrion outer membrane {ECO:0000269|PubM...,MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...,"[GO:0048583, GO:0050789, GO:0010646, GO:002305...","[GO:0008150, GO:0050789, GO:0065007, GO:004858...","[GO:0003674, GO:0005488, GO:0005515]","[GO:0005575, GO:0110165, GO:0005622, GO:004322...",AF-A0A0B4J2F0-F1-model_v4.cif,9606.ENSP00000484893,NaN,[IPR057394],"{""IPR057394"": [1, 53]}"
2,A0A0A7EPL0,E4 SUMO-protein ligase PIAL1 (EC 2.3.2.-) (Pro...,E4-type SUMO ligase that promotes SUMO chain f...,Arabidopsis thaliana (Mouse-ear cress),847.0,Nucleus {ECO:0000305}.,MVIPATSRFGFRAEFNTKEFQASCISLANEIDAAIGRNEVPGNIQE...,"[GO:0008152, GO:0036211, GO:0009628, GO:006500...","[GO:0008150, GO:0008152, GO:0065007, GO:004851...","[GO:0003674, GO:0003824, GO:0016740, GO:014009...",None,AF-A0A0A7EPL0-F1-model_v4.cif,3702.A0A0A7EPL0,"[ESD4, SUMO1, ULP1D, ULP1B, SUMO8, SUMO2, MOM1...","[IPR004181, IPR013083]","{""IPR013083"": [255, 363], ""IPR004181"": [268, 3..."
3,A0A0B4J1G0,Low affinity immunoglobulin gamma Fc region re...,Receptor for the invariable Fc fragment of imm...,Mus musculus (Mouse),249.0,"Cell membrane {ECO:0000269|PubMed:16039578, EC...",MWQLLLPTALVLTAFSGIQAGLQKAVVNLDPKWVRVLEEDSVTLRC...,"[GO:0006898, GO:0051234, GO:0002376, GO:004578...","[GO:0008150, GO:0002376, GO:0009987, GO:006500...","[GO:0003674, GO:0060089, GO:0038023, GO:000488...","[GO:0005575, GO:0110165, GO:0071944, GO:001602...",AF-A0A0B4J1G0-F1-model_v4.cif,10090.ENSMUSP00000077873,"[Gm5150, Vav1, Tyrobp, Cd247, Fcgr2b, Lilra6, ...","[IPR036179, IPR007110, IPR013783, IPR003598, I...","{""IPR013783"": [20, 189], ""IPR036179"": [23, 188..."
4,A0A0B4J1N3,Protein GPR15LG (Protein GPR15 ligand) (Protei...,Highly cationic protein that has multiple func...,Mus musculus (Mouse),78.0,Secreted {ECO:0000250|UniProtKB:Q6UWK7}.,MRLLALSGLLCMLLLCFCIFSSEGRRHPAKSLKLRRCCHLSPRSKL...,"[GO:0009605, GO:0051716, GO:0040011, GO:000237...","[GO:0008150, GO:0040011, GO:0002376, GO:000998...","[GO:0003674, GO:0098772, GO:0005488, GO:014067...",None,AF-A0A0B4J1N3-F1-model_v4.cif,10090.ENSMUSP00000136426,[Gpr15],[IPR031713],"{""IPR031713"": [1, 77]}"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133529,V9XTM1,Erythrocyte-binding antigen 175,None,Plasmodium sp. chimpanzee clade C3,631.0,None,FLDSRVNDRKNSSSNNGDLNNCREKRRGMKWECKKKNNTSNYVCIP...,"[GO:0005515, GO:0005488, GO:0003674]",None,"[GO:0003674, GO:0005488, GO:0005515]",None,AF-V9XTM1-F1-model_v4.cif,None,NaN,"[IPR008602, IPR042202]","{""IPR042202"": [6, 488], ""IPR008602"": [43, 503]}"
133530,W5IDC3,Autonomous cohesin,None,Ruminococcus flavefaciens,201.0,None,MGSSSVTADLNNAVINVDEMNEAFKDVPDLEGEGAHITLSNTTAKP...,"[GO:0005515, GO:0005488, GO:0003674]",None,"[GO:0003674, GO:0005488, GO:0005515]",None,AF-W5IDC3-F1-model_v4.cif,None,NaN,None,NaN
133531,Q9YWN5,VP9,None,Banna virus (BAV),283.0,None,MLSETELRALKKLSTTTSRVVGDSTLALPSNVKLSKGEVEKIAVTK...,"[GO:0042802, GO:0005488, GO:0005515, GO:0003674]",None,"[GO:0003674, GO:0005488, GO:0005515, GO:0042802]",None,None,None,NaN,"[IPR043116, IPR015072]","{""IPR043116"": [27, 72], ""IPR01507

### Testing

In [137]:
import pandas as pd

# Combine all sets into one DataFrame
all_sets = pd.concat([cafa5_train, cafa5_test, cafa5_test_super])
all_sets.reset_index(drop=True, inplace=True)

# Create a set of valid (protein, partner) pairs from all_metadata_df
valid_pairs = set(
    zip(all_metadata_df['cafa5_protein'], all_metadata_df['partner_id'])
)

# Flatten and create (protein, partner) pairs from interaction_partners column
errors = []

for idx, row in all_sets.iterrows():
    protein = row['protein_id']
    partners = row['interaction_partners']

    if not isinstance(partners, list):
        if np.isnan(partners):
            continue
        else:
            errors.append((protein, "interaction_partners not a list"))
            continue

    for partner in partners:
        if (protein, partner) not in valid_pairs:
            errors.append((protein, partner))

# Final test result
if not errors:
    print("✅ All interaction_partners are valid and match all_metadata_df.")
else:
    print(f"❌ Found {len(errors)} mismatches. Examples:")
    print(errors[:10])  # Show first 10 mismatches

✅ All interaction_partners are valid and match all_metadata_df.


In [139]:
critical_cols = ['cafa5_protein', 'partner_id', 'string_protein_id', 'Protein names']
missing = all_metadata_df[critical_cols].isnull().sum()
print("Missing values per column:\n", missing)

Missing values per column:
 cafa5_protein        0
partner_id           0
string_protein_id    0
Protein names        0
dtype: int64


In [142]:
assert all(all_metadata_df['cafa5_protein'].apply(lambda x: isinstance(x, str)))
assert all(all_metadata_df['partner_id'].apply(lambda x: isinstance(x, str)))

In [143]:
dups = all_metadata_df.duplicated(subset=['cafa5_protein', 'partner_id'])
print(f"Number of duplicated interactions: {dups.sum()}")

Number of duplicated interactions: 0


In [146]:
all_cafa5_ids = set(pd.concat([cafa5_train, cafa5_test, cafa5_test_super])['protein_id'])
metadata_ids = set(all_metadata_df['cafa5_protein'])

not_in_cafa5 = metadata_ids - all_cafa5_ids
print(f"Number of proteins in all_metadata_df not in CAFA5 sets: {len(not_in_cafa5)}")

Number of proteins in all_metadata_df not in CAFA5 sets: 0


In [147]:
if 'combined_score' in all_metadata_df.columns:
    print(all_metadata_df['combined_score'].describe())
    assert all_metadata_df['combined_score'].between(0, 1000).all()

count    4.000111e+06
mean     8.568001e+02
std      9.526920e+01
min      7.000000e+02
25%      7.710000e+02
50%      8.560000e+02
75%      9.430000e+02
max      9.990000e+02
Name: combined_score, dtype: float64


## Time to Upload

In [149]:
from datasets import Dataset
import json

# Create Hugging Face Datasets
train_hf = Dataset.from_pandas(cafa5_train, preserve_index=False)
test_hf = Dataset.from_pandas(cafa5_test, preserve_index=False)
test_super_hf = Dataset.from_pandas(cafa5_test_super, preserve_index=False)

In [150]:
from datasets import DatasetDict

cafa_dataset = DatasetDict({
    "train": train_hf,
    "test": test_hf,
    "test_superset": test_super_hf
})

In [151]:
cafa_dataset

DatasetDict({
    train: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path', 'string_id', 'interaction_partners', 'interpro_ids', 'interpro_location'],
        num_rows: 133534
    })
    test: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path', 'string_id', 'interaction_partners', 'interpro_ids', 'interpro_location'],
        num_rows: 264
    })
    test_superset: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path', 'string_id', 'interaction_partners', 'interpro_ids', 'interpro_location'],
        num_rows: 141847
    })
})

In [152]:
cafa_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="cafa5_reasoning",
    commit_message="Edited CAFA data to reflect PPI metadata"
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/134 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/147M [00:00<?, ?B/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/304k [00:00<?, ?B/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/142 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/120M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/14f28777af9547f1f51d12a8786f1eb8d57bafb8', commit_message='Edited CAFA data to reflect PPI metadata', commit_description='', oid='14f28777af9547f1f51d12a8786f1eb8d57bafb8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)

In [153]:
from datasets import Dataset
import json

# Create Hugging Face Datasets
ppi_metadata = Dataset.from_pandas(all_metadata_df, preserve_index=False)

In [154]:
from datasets import DatasetDict

ppi_dataset = DatasetDict({
    "metadata": ppi_metadata
})

In [155]:
ppi_dataset

DatasetDict({
    metadata: Dataset({
        features: ['cafa5_protein', 'partner_id', 'string_protein_id', 'uniprot_ac', 'uniprot_id', 'identity', 'bit_score', 'coexpression', 'combined_score', 'cooccurence', 'database', 'experimental', 'fusion', 'neighborhood', 'protein2', 'textmining', 'Protein names', 'Function [CC]', 'Subcellular location [CC]'],
        num_rows: 4000111
    })
})

In [156]:
ppi_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="ppi_metadata",
    commit_message="Uploading all of the PPI metadata to one central place. Indexed by cafa5_protein and interaction partner"
)

Uploading the dataset shards:   0%|          | 0/7 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/572 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/206M [00:00<?, ?B/s]

Creating parquet from Arrow format:   0%|          | 0/572 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/202M [00:00<?, ?B/s]

...

Uploading...:   0%|          | 0.00/58.4M [00:00<?, ?B/s]

Creating parquet from Arrow format:   0%|          | 0/572 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/89.4M [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/bdb765ec26f087906554623cab76d8af116bcc97', commit_message='Uploading all of the PPI metadata to one central place. Indexed by cafa5_protein and interaction partner', commit_description='', oid='bdb765ec26f087906554623cab76d8af116bcc97', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)